# NLP project: attention heads across model collapse

Plots of the attention heads of the collapse runs in `results/`:

1. the previous-token and induction score of every head in each generation, next to the heads in the discovered ACDC circuit,
2. the strongest induction score of each generation for every condition of the 20-generation runs at T = 3.

Run `bash scripts/nlp_project_run.sh` first. The notebook reads every collapse run that exists; sections whose runs are missing are skipped. Figures are saved to `results/figures/nlp_project/`.

Use the `.venv` Python kernel. Only JAX is needed, not PyTorch.

In [ ]:
# Locate the project when launched from the root or a plotting subfolder.
from pathlib import Path
import os
import sys

PROJECT_ROOT = next(
    p for p in (Path.cwd(), *Path.cwd().parents)
    if (p / "src" / "main.py").is_file() and (p / "requirements.txt").is_file()
)
sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

import glob
import json
from copy import deepcopy
from functools import partial

import numpy as np
import h5py as h5
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

import jax
import jax.numpy as jnp
import equinox as eqx

from src import generations, main, main_utils, opto

%matplotlib inline

In [ ]:
base_folder = './results'
fig_folder = f'{base_folder}/figures/nlp_project'
os.makedirs(fig_folder, exist_ok=True)

N_SEQ = 1024                # real training sequences used for every plot
SEQ_SEED = 7                # the circuit discovery prompts use the same seed

def newest_run(name):
    runs = sorted(glob.glob(f'{base_folder}/*_{name}'))
    runs = [r for r in runs if os.path.isdir(r)]
    return runs[-1] if runs else None


In [ ]:
# Plot style: recessive axes, single-hue attention maps, a diverging map for signed scores.
matplotlib.rcParams.update({
    'font.size': 11, 'axes.titlesize': 12, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.edgecolor': '#8a8984', 'axes.labelcolor': '#52514e', 'xtick.color': '#52514e',
    'ytick.color': '#52514e', 'axes.grid': False, 'figure.dpi': 110, 'savefig.bbox': 'tight',
})
ATTN_CMAP = LinearSegmentedColormap.from_list('attn', ['#fcfcfb', '#9ec5f0', '#2a78d6', '#0d3a73'])
SCORE_CMAP = LinearSegmentedColormap.from_list('score', ['#eb6834', '#f3f2ee', '#2a78d6'])
SERIES = ['#2a78d6', '#eb6834', '#1baf7a', '#eda100']  # categorical slots, fixed order
MUTED = '#c3c2b7'

def save(fig, name):
    for ext in ('png', 'pdf'):
        fig.savefig(f'{fig_folder}/{name}.{ext}')

## Loading runs and extracting attention

A sequence of context length 2 is `x1 y1 x2 y2 xq`: exemplar and label pairs, then the query exemplar, whose label the model predicts. The extended model's sequences continue with the query label `yq` and the next symbol `xn`; it predicts the next symbol at `yq`.

The two head scores follow Singh et al. (2024):

* **previous-token score**: the attention each label position `yi` gives to its own exemplar `xi`, the step before it;
* **induction score**: the attention the query `xq` gives to the label of the matching exemplar, minus the attention it gives to the other label. A score of 1 means the head always looks up the correct label.

In [ ]:
class Run:
    """A trained run: its options, data, checkpoints and a fixed set of real sequences."""

    def __init__(self, folder, n_seq=N_SEQ, seed=SEQ_SEED):
        self.folder = folder
        argv, sys.argv = sys.argv, sys.argv[:1]  # the option parser reads sys.argv
        try:
            self.opts = main_utils.get_opts_from_json_file(f'{folder}/config.json')
        finally:
            sys.argv = argv
        self.data = main_utils.get_data_from_opts(self.opts)
        self.splits = main_utils.get_splits_from_opts(self.opts, self.data.shape)
        self.extended = self.opts.model_variant == 'extended'
        model = main_utils.get_model_from_opts(self.opts, (self.data.shape[-1],))
        opt_state = main_utils.get_optimizer_from_opts(self.opts).init(eqx.filter(model, eqx.is_array))
        key = jax.random.PRNGKey(0)
        self.ckpt_fmt = {'iter': -1, 'opt_state': opt_state, 'model': model,
                         'seeds': {'eval_model_seed': key, 'train_data_seed': key, 'train_model_seed': key}}
        sampler = main.make_train_data_sampler(self.opts, self.data, self.splits)
        self.seqs = generations.build_pool(sampler, n_seq, jax.random.PRNGKey(seed))
        self.n_ctx = self.seqs['labels'].shape[1] - (2 if self.extended else 1)
        self.tokens = self.token_names()

    def token_names(self):
        names = []
        for i in range(self.n_ctx):
            names += [f'x{i + 1}', f'y{i + 1}']
        names += ['xq', 'yq', 'xn'] if self.extended else ['xq']
        return names

    def checkpoints(self, gen_folder=None):
        folder = f'{gen_folder or self.folder}/checkpoints'
        files = sorted(os.listdir(folder))
        return [f'{folder}/{f}' for f in files], np.array([int(f.split('.')[0]) for f in files])

    def load(self, path=None):
        """Loads a checkpoint, by default the last one of the run."""
        path = path or self.checkpoints()[0][-1]
        return eqx.tree_deserialise_leaves(path, self.ckpt_fmt)['model']


@eqx.filter_jit
def _forward(model, x, y):
    keys = jax.random.split(jax.random.PRNGKey(0), x.shape[0])
    r = jax.vmap(partial(opto.default_model_fwd_fn, model=model))(x=x, y=y, key=keys)
    attn = jnp.stack([b['attn_output']['attn_scores'][:, 0] for b in r['transformer_output']['block_outputs']], 1)
    return attn, r['out']


def attention(run, model):
    """Attention patterns, shape (sequence, layer, head, query position, key position)."""
    attn, _ = _forward(model, run.seqs['examples'], run.seqs['labels'])
    return np.asarray(attn)


def head_scores(run, attn):
    """Previous-token and induction score of every head, each shape (layer, head)."""
    n = run.n_ctx
    labels_pos = 2 * np.arange(n) + 1
    prev = attn[:, :, :, labels_pos, labels_pos - 1].mean(axis=(0, 3))
    s = np.asarray(run.seqs['symbols'])
    match = s[:, :n] == s[:, [n]]               # which context items match the query
    keep = match.any(1) & (~match).any(1)       # sequences with a correct and a wrong label
    q = attn[keep][:, :, :, 2 * n, labels_pos]  # query attention to every context label
    m = match[keep][:, None, None, :]
    correct = (q * m).sum(-1) / m.sum(-1)
    wrong = (q * ~m).sum(-1) / (~m).sum(-1)
    return prev, (correct - wrong).mean(0)


## 1. Heads across model collapse generations

For each collapse run, every generation's final model is analysed on the same real sequences. The heatmaps show how the previous-token and induction scores of each head change from generation 0 (real data) to the last generation. Where circuit discovery ran, the right panel marks the heads that appear in the ACDC circuit of each generation for the exemplar-swap target.

In [ ]:
def generation_folders(run_folder):
    gens = sorted(glob.glob(f'{run_folder}/gen_*'), key=lambda p: int(p.rsplit('_', 1)[1]))
    return [g for g in gens if os.path.isdir(f'{g}/checkpoints') and os.listdir(f'{g}/checkpoints')]

def acdc_heads(gen_folder, n_layers, n_heads, target='value_exemplar_swap'):
    """1 for every head with an edge in the ACDC circuit, or None without circuit outputs."""
    path = f'{gen_folder}/circuit/{target}_acdc_edges.json'
    if not os.path.exists(path):
        return None
    used = np.zeros((n_layers, n_heads))
    for e in json.load(open(path)):
        if e['in_circuit']:
            for node in (e['src'], e['dest']):
                if node.startswith('A'):
                    l, h = node[1:].split('.')[:2]
                    used[int(l), int(h)] = 1
    return used

def plot_collapse_run(run_folder):
    gens = generation_folders(run_folder)
    if len(gens) < 2:
        return
    run = Run(run_folder)
    prev, ind, circ = [], [], []
    for g in gens:
        model = run.load(run.checkpoints(g)[0][-1])
        p, s = head_scores(run, attention(run, model))
        prev.append(p)
        ind.append(s)
        circ.append(acdc_heads(g, *p.shape))
    prev, ind = np.stack(prev), np.stack(ind)
    has_circ = all(c is not None for c in circ)
    n_heads = prev.shape[2]
    panels = [(prev[:, 0], 'layer 0 previous-token score', ATTN_CMAP, (0, 1)),
              (ind[:, -1], 'last-layer induction score', SCORE_CMAP, (-1, 1))]
    if has_circ:
        panels.append((np.stack([c[-1] for c in circ]), 'last-layer head in ACDC circuit', ATTN_CMAP, (0, 1)))
    fig, axs = plt.subplots(1, len(panels), figsize=(5.2 * len(panels), 0.9 + 0.45 * len(gens)))
    for ax, (values, label, cmap, lims) in zip(axs, panels):
        ax.imshow(values, cmap=cmap, vmin=lims[0], vmax=lims[1], aspect='auto')
        for g in range(values.shape[0]):
            for h in range(n_heads):
                ax.text(h, g, f'{values[g, h]:.2f}' if 'circuit' not in label else ('in' if values[g, h] else ''),
                        ha='center', va='center', fontsize=8,
                        color='white' if abs(values[g, h]) > 0.6 else '#0b0b0b')
        ax.set_xticks(range(n_heads), [f'H{h}' for h in range(n_heads)])
        ax.set_yticks(range(len(gens)), [f'gen {g}' for g in range(len(gens))])
        ax.tick_params(length=0)
        for side in ax.spines.values():
            side.set_visible(False)
        ax.set_title(label)
    name = os.path.basename(run_folder).split('_', 1)[1]
    fig.suptitle(f'{name}: head scores per generation')
    fig.tight_layout()
    save(fig, f'{name}_heads_per_generation')
    plt.show()

collapse_runs = sorted(r for r in glob.glob(f'{base_folder}/*_collapse_*') if os.path.isdir(r))
print(f'{len(collapse_runs)} collapse runs found')
for r in collapse_runs:
    plot_collapse_run(r)

## 2. Induction score of the strongest head across conditions

One line per condition of the 20-generation runs at T = 3 (first seed): the largest last-layer induction score of each generation. The control (`real_only`) trains on real data in every generation.

In [ ]:
def strongest_induction_per_generation(run_folder):
    run = Run(run_folder)
    return [float(head_scores(run, attention(run, run.load(run.checkpoints(g)[0][-1])))[1][-1].max())
            for g in generation_folders(run_folder)]

CONDITIONS = {'real_only': 'G20_real_only', 'replace50': 'G20_T3_replace50', 'replace90': 'G20_T3_replace90',
              'replace100': 'G20_T3_replace100', 'accumulate': 'G20_T3_accumulate'}  # condition: run name
for variant in ['base', 'extended']:
    found = {c: newest_run(f'collapse_{variant}_{run}') for c, run in CONDITIONS.items()}
    found = {c: r for c, r in found.items() if r and len(generation_folders(r)) > 1}
    if not found:
        continue
    fig, ax = plt.subplots(figsize=(8, 4.2))
    colours = dict(zip(CONDITIONS, [MUTED] + SERIES))  # the control is grey
    for c, r in found.items():
        values = strongest_induction_per_generation(r)
        ax.plot(range(len(values)), values, color=colours[c], lw=2, marker='o', ms=8,
                markeredgecolor='white', markeredgewidth=2, label=c)
    ax.xaxis.set_major_locator(matplotlib.ticker.MaxNLocator(integer=True))
    ax.set_xlabel('generation')
    ax.set_ylabel('strongest induction score')
    ax.set_ylim(-0.05, 1.05)
    ax.set_title(f'{variant} model: induction head strength across generations')
    ax.legend(frameon=False, fontsize=9)
    save(fig, f'collapse_{variant}_induction_across_generations')
    plt.show()